In [1]:
# %pip install -q langchain-ollama

In [2]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2:1b")  
llm.invoke("What is the capital of France?") # 프롬프트 : LLM호출 

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-13T03:20:08.3080711Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3235257900, 'load_duration': 3152701500, 'prompt_eval_count': 32, 'prompt_eval_duration': 14425900, 'eval_count': 8, 'eval_duration': 60017100, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019ce535-5ace-7710-aa0c-474610a13a51-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser  # output parser

prompt_template = PromptTemplate(
    template = "What is the capital of {country}?",   
    input_variables=["country"],
)

prompt = prompt_template.invoke({"country":"France"})  
output_parser = StrOutputParser()  
answer = output_parser.invoke(llm.invoke(prompt))
answer

'The capital of France is Paris.'

In [4]:
# chain

# invoke 반대로 연결 
capital_chain = prompt_template | llm | output_parser  # 가독성이 더 좋다

# chain 도 Runnable이기 때문에 invoke가능
capital_chain.invoke({"country" : "France"})

'The capital of France is Paris.'

In [5]:
## chain 간에도 연결 가능하다 

# 나라 정보를 찾는 chain을 생성해서 수도 chain과 연결하는 코드

country_prompt = PromptTemplate(
    template="""Guess the name of the country based on the following information:
    {information}
    Return the name of the country only
    """,
    input_variables=["information"],

)


country_chain = country_prompt | llm | output_parser
country_chain.invoke({"information":"This country is famous for its wine in Europe"})

'Italy'

In [6]:
# 위 2개의 chain으로 최종 chain을 만든다

final_chain = {"country" : country_chain} | capital_chain  # country_chain의 결과를 capital_chain에 준다
# 그런데 capital_chain을 보면, country가 필요하다  {"country" : country_chain} 로 변환해야 함

In [7]:
# final_chain의 첫 단계가 country_chain이기 때문에, 입력은 country_chain이 요구하는 형식이어야 함
final_chain.invoke({"information":"This country is famous for its wine in Europe"})

'The capital of Spain is Madrid.'

- RunnablePassThrough

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# RunnablePassthrough : invoke 할때 "information" 키값을 안줘도 동작한다
final_chain = {"information" : RunnablePassthrough()} | {"country" : country_chain} | capital_chain  # country_chain의 결과를 capital_chain에 준다
final_chain.invoke("This country is famous for its wine in Europe")

# 플로우 : This country is famous for its wine in Europe 쿼리 -> "information" 키로 들어가서 -> country_chain invoke로 들어간다 -> 이를 기반으로 country_chain을 실행하고 -> 그 답변이 country가 되어 capital_chain에 들어간다 -> 최종적으로

- RunnablePassThrough의 값이 2개일 경우

In [8]:
## chain 간에도 연결 가능하다 

# 나라 정보를 찾는 chain을 생성해서 수도 chain과 연결하는 코드

country_prompt = PromptTemplate(
    template="""Guess the name of the country in {continent} based on the following information:
    {information}
    Return the name of the country only
    """,
    input_variables=["information"],

)


country_chain = country_prompt | llm | output_parser

In [10]:
from langchain_core.runnables import RunnablePassthrough

# invoke 문에 키를 직접 지정해줘야 한다.
final_chain = {"information" : RunnablePassthrough(), "continent" : RunnablePassthrough()} | {"country" : country_chain} | capital_chain  # country_chain의 결과를 capital_chain에 준다
final_chain.invoke({"information" : "This country is famous for its wine in Europe", "continent" : "Europe"})

'The capital of Italy is Rome (Italian: Roma).'